In [1]:
import pandas as pd

# Load the final dataset
final_df = pd.read_csv("final_ds.csv")

# Load the metadata CSV containing image filenames
metadata_df = pd.read_csv("satellite_images/satellite_images_metadata.csv")

# If your final_df does not have an explicit event identifier,
# add one based on its index.
if 'event_id' not in final_df.columns:
    final_df["event_id"] = final_df.index

# Group metadata by event_id and aggregate image filenames into a list.
filenames_by_event = metadata_df.groupby("event_id")["image_filename"].apply(list).reset_index()
filenames_by_event.rename(columns={"image_filename": "filenames"}, inplace=True)
# print(filenames_by_event)
# Merge the aggregated filenames with the final dataset.
merged_df = pd.merge(final_df, filenames_by_event, on="event_id", how="left")
merged_df.head()
# # Optionally, if you don't need the event_id column, you can drop it:
# # merged_df.drop("event_id", axis=1, inplace=True)

# # Save the merged DataFrame to a new CSV
# merged_df.to_csv("final_ds_with_filenames.csv", index=False)

# print("Merged dataset saved as final_ds_with_filenames.csv")


,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,begin_time_utc,end_date_utc,end_time_utc,cluster,start_date_72h,end_date,event_datetime,prev_72h_weather,event_id,filenames
0,Flood,2012-10-29 12:30:00,EST-5,2012-10-29 22:30:00,37.84,-75.48,37.70,-75.62,0,2012-10-29,17:30:00,2012-10-30,03:30:00,4,2012-10-26,2012-10-29,2012-10-29 17:30:00+00:00,[{'date': Timestamp('2012-10-26 18:00:00+0000'...,0,[event0_Flood_IMERG_Precipitation_Rate_2012-10...
1,Heavy Rain,2014-05-15 14:00:00,EST-5,2014-05-16 17:00:00,37.84,-75.48,37.82,-75.62,0,2014-05-15,19:00:00,2014-05-16,22:00:00,4,2014-05-12,2014-05-15,2014-05-15 19:00:00+00:00,[{'date': Timestamp('2014-05-12 19:00:00+0000'...,1,[event1_Heavy_Rain_IMERG_Precipitation_Rate_20...
2,Heavy Rain,2016-10-08 10:00:00,EST-5,2016-10-09 11:00:00,37.84,-75.48,37.84,-75.48,0,2016-10-08,15:00:00,2016-10-09,16:00:00,4,2016-10-05,2016-10-08,2016-10-08 15:00:00+00:00,[{'date': Timestamp('2016-10-05 15:00:00+0000'...,2,[event2_Heavy_Rain_IMERG_Precipitation_Rate_20...
3,Flash Flood,2014-09-21 23:14:00,MST-7,2014-09-22 02:00:00,32.90,-107.34,32.82,-107.34,0,2014-09-22,06:14:00,2014-09-22,09:00:00,19,2014-09-18,2014-09-21,2014-09-22 06:14:00+00:00,[{'date': Timestamp('2014-09-19 07:00:00+0000'...,3,[event3_Flash_Flood_IMERG_Precipitation_Rate_2...
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,23:00:00,2014-09-28,06:00:00,48,2014-09-24,2014-09-27,2014-09-27 23:00:00+00:00,[{'date': Timestamp('2014-09-24 23:00:00+0000'...,4,[event4_Flash_Flood_IMERG_Precipitation_Rate_2...


In [2]:
merged_df = merged_df.iloc[:7000, :]

In [3]:
import re
import pandas as pd

def func(s):
    if not isinstance(s, str):
        return []  # Return empty list for non-string inputs
    
    dict_pattern = r'\{[^}]+\}'
    dict_matches = re.findall(dict_pattern, s)
    kv_pattern = r"'([^']+)':\s([^,}]+)"
    extracted_data = []

    for dict_str in dict_matches:  # Limit to first 10 dictionaries
        kv_matches = re.findall(kv_pattern, dict_str)
        extracted_dict = {key: value.strip() for key, value in kv_matches}
        extracted_data.append(extracted_dict)
    
    return extracted_data
result = []
# Read the CSV in chunks
chunk_size = 10000  # Adjust this based on your available memory
for chunk in pd.read_csv("final_ds_with_filenames.csv", chunksize=chunk_size):
    chunk["prev_72h_weather"] = chunk['prev_72h_weather'].apply(func)
    result.append(chunk)
    # Process or save the chunk here
    print("Processed chunk")
merged_df = pd.concat(result, ignore_index=True)
merged_df = merged_df.iloc[:7000, :]
print("All chunks processed")


Processed chunk
Processed chunk
Processed chunk
Processed chunk
All chunks processed


In [ ]:
len(merged_df["prev_72h_weather"].iloc[0])

72

In [6]:
import tqdm

tqdm.tqdm.pandas()
def expand_weather_data(row):
    weather_data = row['prev_72h_weather']
    for key in weather_data[0].keys():
        if key != 'date':
            row[f'prev_72h_{key}'] = [entry[key] for entry in weather_data]
    return row

merged_df = merged_df.progress_apply(expand_weather_data, axis=1)


100%|██████████| 7000/7000 [04:29<00:00, 25.99it/s] 


In [7]:
import ast

def clean_and_filter_layers(df, column_name='filename'):
    # Step 1: Convert string representation of list to actual list if needed
    def parse_if_string(x):
        if isinstance(x, str):
            try:
                return ast.literal_eval(x)
            except:
                return x
        return x
    
    # Apply parsing if the column contains string representations of lists
    df[column_name] = df[column_name].progress_apply(parse_if_string)
    
    # Step 2: Filter for specific layers
    patterns = [
        'VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11',
        'VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR',
        'MODIS_Terra_CorrectedReflectance_TrueColor'
    ]
    
    # If the column contains lists, we need to filter elements within each list
    def filter_specific_layers(file_list):
        if isinstance(file_list, list):
            return [f for f in file_list if any(pattern in f for pattern in patterns)]
        return file_list
    
    df[column_name] = df[column_name].progress_apply(filter_specific_layers)
    
    # Remove rows where the filtered list is empty
    df = df[df[column_name].progress_apply(lambda x: len(x) > 0 if isinstance(x, list) else True)]
    
    return df

filtered_df = clean_and_filter_layers(merged_df, column_name='filenames')

100%|██████████| 7000/7000 [00:00<00:00, 478575.49it/s]


In [8]:
filtered_df = filtered_df.iloc[:7000,:]

In [9]:
filtered_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   event_type                              7000 non-null   object 
 1   begin_date_time                         7000 non-null   object 
 2   cz_timezone                             7000 non-null   object 
 3   end_date_time                           7000 non-null   object 
 4   begin_lat                               7000 non-null   float64
 5   begin_lon                               7000 non-null   float64
 6   end_lat                                 7000 non-null   float64
 7   end_lon                                 7000 non-null   float64
 8   extreme                                 7000 non-null   int64  
 9   begin_date_utc                          7000 non-null   object 
 10  begin_time_utc                          7000 non-null   obje

In [10]:
from PIL import Image
import numpy as np
from pathlib import Path

# Function to check if an image is empty (all pixels are either 0 or 255)
def is_image_empty(img_path, threshold=0.70):
    """
    Check if an image is empty or non-informative.
    Args:
        img_path: Path to the image file
        threshold: Percentage of same-colored pixels to consider image empty (0.0 to 1.0)
    Returns:
        bool: True if image is considered empty/non-informative
    """
    try:
        with Image.open(img_path) as img:
            # Convert to grayscale to simplify analysis
            gray_img = img.convert('L')
            img_array = np.array(gray_img)
            
            # Check if image is predominantly white
            white_ratio = np.sum(img_array > 250) / img_array.size
            # Check if image is predominantly black
            black_ratio = np.sum(img_array < 5) / img_array.size
            
            return white_ratio > threshold or black_ratio > threshold
            
    except Exception as e:
        print(f"Error processing {img_path}: {e}")
        return True  # Consider unreadable images as empty

def check_and_filter_empty_images(df, base_path, column_name='filenames', threshold=0.70):
    """
    Check filtered images for emptiness and remove rows containing any empty images.
    
    Args:
        df: DataFrame containing lists of image filenames
        base_path: Base directory path where images are stored
        column_name: Name of the column containing image filenames
        threshold: Threshold for determining empty images
        
    Returns:
        DataFrame with rows removed where any image in the list is empty
    """
    def check_image_list(image_list):
        for img_name in image_list:
            img_path = Path(base_path) / img_name
            if is_image_empty(img_path, threshold):
                return False  # If any image is empty, return False
        return True  # All images are valid
    
    # Apply the check to each row and keep only rows where all images are valid
    mask = df[column_name].progress_apply(check_image_list)
    filtered_df = df[mask]
    
    # Print statistics
    removed_count = len(df) - len(filtered_df)
    print(f"Removed {removed_count} rows containing empty images")
    print(f"Remaining rows: {len(filtered_df)}")
    
    return filtered_df

# Usage example:

In [11]:
# Assuming you have already filtered for the three specific layers
base_path = "satellite_images"  # Replace with your actual path
final_df = check_and_filter_empty_images(filtered_df, base_path)

100%|██████████| 7000/7000 [12:06<00:00,  9.63it/s]


Removed 136 rows containing empty images
Remaining rows: 6864


In [12]:
final_df["extreme"].value_counts()  

extreme
0    6427
1     437
Name: count, dtype: int64

In [13]:
final_df.to_csv("final_filtered_df.csv", index=False)

In [ ]:
import pandas as pd
final_df = pd.read_csv("final_filtered_df.csv") 

In [14]:
len(final_df["prev_72h_wind_direction_100m"].iloc[0].split(","))

AttributeError: 'list' object has no attribute 'split'

In [4]:
final_df["filenames"].iloc[0]

"['event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11_2012-10-29.png', 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR_2012-10-29.png', 'event0_Flood_MODIS_Terra_CorrectedReflectance_TrueColor_2012-10-29.png', 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11_2012-10-28.png', 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR_2012-10-28.png', 'event0_Flood_MODIS_Terra_CorrectedReflectance_TrueColor_2012-10-28.png', 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_SWIR_M11_2012-10-27.png', 'event0_Flood_VIIRS_SNPP_Cirrus_Reflectance_VIS_NIR_2012-10-27.png', 'event0_Flood_MODIS_Terra_CorrectedReflectance_TrueColor_2012-10-27.png']"

## EDA

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import ast

# Step 1: Clean the 'prev_' columns
def clean_prev_column(column):
    def safe_convert(value):
        try:
            if isinstance(value, str):
                return ast.literal_eval(value)
            elif isinstance(value, list):
                return value
            else:
                return []  # Replace invalid entries with an empty list
        except:
            return []  # Handle any parsing errors

    return column.apply(safe_convert)

# Clean all columns starting with 'prev_'
prev_columns = [col for col in df.columns if col.startswith('prev_')]
for col in prev_columns:
    df[col] = clean_prev_column(df[col])

# Step 2: Perform Mann-Whitney U test with balanced sampling
results = {}

for event in ['Flood', 'Heavy Rain', 'Flash Flood', 'Debris Flow']:
    results[event] = {}
    
    # Filter data for this event type
    event_df = df[df['event_type'] == event]
    extreme_event_df = event_df[event_df['extreme'] == 1]
    non_extreme_event_df = event_df[event_df['extreme'] == 0]

    print(f"\nEvent Type: {event}")
    
    for col in prev_columns:
        # Flatten lists into single arrays for comparison
        extreme_values = np.concatenate(extreme_event_df[col].values).astype(float)
        non_extreme_values = np.concatenate(non_extreme_event_df[col].values).astype(float)

        # Remove NaN or invalid values from both arrays
        extreme_values = extreme_values[~np.isnan(extreme_values)]
        non_extreme_values = non_extreme_values[~np.isnan(non_extreme_values)]

        # Balance sampling (sample equal sizes from both groups)
        min_size = min(len(extreme_values), len(non_extreme_values))
        if min_size > 0:
            extreme_sample = np.random.choice(extreme_values, min_size, replace=False)
            non_extreme_sample = np.random.choice(non_extreme_values, min_size, replace=False)

            # Perform Mann-Whitney U test (use exact method when appropriate)
            method = 'exact' if min_size < 50 else 'asymptotic'
            stat, p_value = mannwhitneyu(extreme_sample, non_extreme_sample, alternative='two-sided', method=method)
            results[event][col] = {
                'p_value': p_value,
                'significant': p_value < 0.05
            }
        else:
            results[event][col] = {'p_value': None, 'significant': False}

# Step 3: Display significant results clearly
for event_type, cols in results.items():
    print(f"\nEvent Type: {event_type}")
    for col_name, result in cols.items():
        if result['significant']:
            print(f"Column: {col_name}, p-value: {result['p_value']:.4f} --> Significant difference detected")
        else:
            print(f"Column: {col_name}, p-value: {result['p_value']:.4f} --> No significant difference")



Event Type: Flood

Event Type: Heavy Rain

Event Type: Flash Flood

Event Type: Debris Flow

Event Type: Flood
Column: prev_72h_temperature_2m, p-value: 0.1273 --> No significant difference
Column: prev_72h_relative_humidity_2m, p-value: 0.1718 --> No significant difference
Column: prev_72h_dew_point_2m, p-value: 0.0750 --> No significant difference
Column: prev_72h_apparent_temperature, p-value: 0.0425 --> Significant difference detected
Column: prev_72h_precipitation, p-value: 0.1070 --> No significant difference
Column: prev_72h_rain, p-value: 0.0000 --> Significant difference detected
Column: prev_72h_snowfall, p-value: 0.0000 --> Significant difference detected
Column: prev_72h_snow_depth, p-value: 0.0000 --> Significant difference detected
Column: prev_72h_weather_code, p-value: 0.0099 --> Significant difference detected
Column: prev_72h_pressure_msl, p-value: 0.0000 --> Significant difference detected
Column: prev_72h_surface_pressure, p-value: 0.0000 --> Significant difference

In [1]:
import pandas as pd
import numpy as np
import ast

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, f1_score, 
                             precision_score, recall_score, confusion_matrix, make_scorer)
from sklearn.preprocessing import StandardScaler

# Import SMOTE from imblearn for oversampling
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as imbpipeline

# Assume 'df' is your DataFrame with 7000 rows.
df = pd.read_csv("final_filtered_df.csv")
df.drop(columns=["prev_72h_weather"], inplace=True)
extreme = df[df["extreme"] == 1]
non_extreme = df[df["extreme"] == 0].sample(437)
df = pd.concat([extreme, non_extreme])
columns_to_drop = [f'img_feat_{i}' for i in range(882)]
columns_to_drop.append('prev_72h_weather"')
# Drop the columns from the dataframe
df = df.drop(columns=columns_to_drop, errors='ignore')

# If you want to see the remaining columns

# If the 'prev_' columns are string representations of lists, convert them:
def safe_literal_eval(x):
    if isinstance(x, list):
        x = [float(i) for i in x]
        return x
    try:

        temp = ast.literal_eval(x)
        temp = [float(i) for i in temp]
        return temp

    except Exception as e:
        return np.nan

# Identify all columns starting with "prev_"
prev_cols = [col for col in df.columns if col.startswith('prev_')]
for col in prev_cols:
    df[col] = df[col].apply(safe_literal_eval)

# Aggregate each "prev_" column into summary statistics: mean, std, min, and max.
for col in prev_cols:
    df[f'{col}_mean'] = df[col].apply(lambda x: np.mean(x) if isinstance(x, list) else np.nan)
    df[f'{col}_std'] = df[col].apply(lambda x: np.std(x) if isinstance(x, list) else np.nan)
    df[f'{col}_min'] = df[col].apply(lambda x: np.min(x) if isinstance(x, list) else np.nan)
    df[f'{col}_max'] = df[col].apply(lambda x: np.max(x) if isinstance(x, list) else np.nan)

# Drop the raw list columns to simplify the data
df.drop(columns=prev_cols, inplace=True)

# Drop columns that are non-numeric or not needed for modeling
cols_to_drop = ['event_type', 'begin_date_time', 'cz_timezone', 'end_date_time',
                'begin_date_utc', 'begin_time_utc', 'end_date_utc', 'end_time_utc',
                'start_date_72h', 'end_date', 'event_datetime', 'prev_72h_weather',
                'filenames']
df_model = df.drop(columns=cols_to_drop, errors='ignore')

# Separate features (X) and target (y). Here, 'extreme' is the target.
X = df_model.drop(columns=['extreme', 'cluster', 'event_id'], errors='ignore')
y = df_model['extreme']

# Fill any missing values with the column mean.
X = X.fillna(X.mean())

# Split into training and test sets with stratification to maintain class distribution.
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, 
                                                    test_size=0.2, random_state=42)

# Build a pipeline that includes SMOTE for oversampling, scaling, and a classifier.
# We use imblearn's Pipeline to incorporate the sampler.
pipeline = imbpipeline([
    ('sampler', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

# Define scoring metrics
scoring = {
    'roc_auc': 'roc_auc',
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall'
}

# Use stratified 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)

print("Cross-validation Results with SMOTE:")
for key in sorted(cv_results.keys()):
    print(f"{key}: {np.mean(cv_results[key]):.4f}")

# Hyperparameter tuning via GridSearchCV (including resampling in the pipeline)
param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20]
}

grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)
print("\nBest hyperparameters with SMOTE:")
print(grid.best_params_)
print(f"Best CV ROC AUC: {grid.best_score_:.4f}")

# Evaluate the best model on the test set
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
bal_acc = balanced_accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\nTest Set Evaluation Metrics with SMOTE:")
print(f"ROC AUC: {roc_auc:.4f}")
print(f"Balanced Accuracy: {bal_acc:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("Confusion Matrix:")
print(cm)


C:\Users\naman\AppData\Local\Temp\ipykernel_24756\4130433107.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_min'] = df[col].apply(lambda x: np.min(x) if isinstance(x, list) else np.nan)
C:\Users\naman\AppData\Local\Temp\ipykernel_24756\4130433107.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_max'] = df[col].apply(lambda x: np.max(x) if isinstance(x, list) else np.nan)
C:\Users\naman\AppData\Local\Temp\ipykernel_24756\4130433107.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is

Cross-validation Results with SMOTE:
fit_time: 1.1134
score_time: 0.0739
test_balanced_accuracy: 0.6509
test_f1: 0.6487
test_precision: 0.6526
test_recall: 0.6514
test_roc_auc: 0.7263
train_balanced_accuracy: 1.0000
train_f1: 1.0000
train_precision: 1.0000
train_recall: 1.0000
train_roc_auc: 1.0000

Best hyperparameters with SMOTE:
{'clf__max_depth': 10, 'clf__n_estimators': 200}
Best CV ROC AUC: 0.7342

Test Set Evaluation Metrics with SMOTE:
ROC AUC: 0.7534
Balanced Accuracy: 0.6462
F1 Score: 0.6737
Precision: 0.6214
Recall: 0.7356
Confusion Matrix:
[[49 39]
 [23 64]]


In [ ]:
import satlaspretrain_models
import torch

weights_manager = satlaspretrain_models.Weights()
model = weights_manager.get_pretrained_model(model_identifier="Sentinel2_SwinB_SI_RGB", device="cpu")

# Load the model weights onto the CPU

model.eval()

Model(
  (backbone): SwinBackbone(
    (backbone): SwinTransformer(
      (features): Sequential(
        (0): Sequential(
          (0): Conv2d(3, 128, kernel_size=(4, 4), stride=(4, 4))
          (1): Permute()
          (2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (1): Sequential(
          (0): SwinTransformerBlockV2(
            (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
            (attn): ShiftedWindowAttentionV2(
              (qkv): Linear(in_features=128, out_features=384, bias=True)
              (proj): Linear(in_features=128, out_features=128, bias=True)
              (cpb_mlp): Sequential(
                (0): Linear(in_features=2, out_features=512, bias=True)
                (1): ReLU(inplace=True)
                (2): Linear(in_features=512, out_features=4, bias=False)
              )
            )
            (stochastic_depth): StochasticDepth(p=0.0, mode=row)
            (norm2): LayerNorm((128,), eps=1e-05, element

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import pandas as pd
import ast
import logging
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

tqdm.pandas()  # Registers progress_apply for pandas

# ==========================
# SET UP LOGGING
# ==========================
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# ==========================
# LOAD DATAFRAME
# ==========================
df = pd.read_csv("final_filtered_df.csv")
extreme = df[df["extreme"] == 1]
non_extreme = df[df["extreme"] == 0].sample(437)
df = pd.concat([extreme, non_extreme])

def safe_literal_eval(x):
    """Safely evaluate a string representation of a list."""
    if isinstance(x, list):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return []
df['filenames'] = df['filenames'].apply(safe_literal_eval)

# ==========================
# DEFINE IMAGE TRANSFORMATIONS
# ==========================
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ==========================
# DEFINE THE ROW PROCESSING FUNCTION
# ==========================
def process_row(filenames):
    """
    Process a list of image filenames (for one row): load images,
    extract features from each, and concatenate them.
    """
    features = []
    for file in filenames:
        try:
            full_path = "satellite_images/" + file
            image = Image.open(full_path).convert('RGB')
            image = preprocess(image)
            image = image.unsqueeze(0)  # Add batch dimension
            with torch.no_grad():
                x = image
                row_features = []
                # Iterate over layers in your model's backbone
                for i, layer in enumerate(model.backbone.backbone.features):
                    x = layer(x)
                    if i in selected_layers:
                        pooled_feature = torch.mean(x, dim=(2, 3))
                        row_features.append(pooled_feature)
                if row_features:
                    concatenated_features = torch.cat(row_features, dim=1)
                    features.append(concatenated_features.squeeze(0).numpy())
        except Exception as e:
            logging.error(f"Error processing file {file}: {e}")
            features.append(np.zeros(98))
    if features:
        return np.concatenate(features)
    else:
        return np.zeros(98)

# ==========================
# LOAD YOUR MODEL
# ==========================
# Make sure your model is loaded and in evaluation mode.
# For example, using timm:
# import timm
# model = timm.create_model('swin_base_patch4_window7_224', pretrained=True)
# model.eval()

try:
    model
except NameError:
    raise NameError("Model is not defined. Please load your pre-trained model before running.")

selected_layers = [1, 3, 5]  # Adjust these indices as needed

# ==========================
# PARALLEL PROCESSING WITH THREADPOOL
# ==========================
logging.info("Starting parallel image feature extraction across rows using threads.")
results = [None] * len(df)  # Preallocate list for ordered results

with ThreadPoolExecutor(max_workers=8) as executor:  # Adjust max_workers if needed
    # Submit one task per row (each row's filenames list)
    future_to_index = {executor.submit(process_row, row): idx for idx, row in enumerate(df['filenames'])}
    for future in tqdm(as_completed(future_to_index), total=len(future_to_index), desc="Rows processed"):
        idx = future_to_index[future]
        try:
            results[idx] = future.result()
        except Exception as e:
            logging.error(f"Error processing row {idx}: {e}")
            results[idx] = np.zeros(98)
            
df['image_features'] = results
logging.info("Completed parallel image feature extraction.")

2025-03-13 10:43:48,807 - INFO - Starting parallel image feature extraction across rows using threads.
Rows processed: 100%|██████████| 874/874 [1:00:30<00:00,  4.15s/it]
2025-03-13 11:44:19,751 - INFO - Completed parallel image feature extraction.


In [ ]:
df["image_features"].iloc[55].shape

(882,)

In [ ]:
df.to_csv("final_df_image_features.csv", index=False)

In [21]:
import pandas as pd
temp = pd.read_csv("final_df_image_features.csv")
temp.drop(columns=["prev_72h_weather"], inplace=True)
temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 874 entries, 0 to 873
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   event_type                              874 non-null    object 
 1   begin_date_time                         874 non-null    object 
 2   cz_timezone                             874 non-null    object 
 3   end_date_time                           874 non-null    object 
 4   begin_lat                               874 non-null    float64
 5   begin_lon                               874 non-null    float64
 6   end_lat                                 874 non-null    float64
 7   end_lon                                 874 non-null    float64
 8   extreme                                 874 non-null    int64  
 9   begin_date_utc                          874 non-null    object 
 10  begin_time_utc                          874 non-null    object

In [22]:
import numpy as np
import ast
temp.drop
def safe_literal_eval(x):
    if isinstance(x, list):
        x = [float(i) for i in x]
        return x
    try:
        temp = ast.literal_eval(x)
        temp = [float(i) for i in temp]
        return temp

    except Exception as e:
        print(e, x)
        return np.nan
prev_cols = [col for col in temp.columns if col.startswith('prev_')]
for col in prev_cols:
    temp[col] = temp[col].apply(safe_literal_eval)



In [23]:
temp

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_wind_gusts_10m,prev_72h_soil_temperature_0_to_7cm,prev_72h_soil_temperature_7_to_28cm,prev_72h_soil_temperature_28_to_100cm,prev_72h_soil_temperature_100_to_255cm,prev_72h_soil_moisture_0_to_7cm,prev_72h_soil_moisture_7_to_28cm,prev_72h_soil_moisture_28_to_100cm,prev_72h_soil_moisture_100_to_255cm,image_features
0,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,...,"[26.639999389648438, 24.119998931884766, 19.79...","[30.48900032043457, 28.388999938964844, 25.088...","[19.68899917602539, 19.93899917602539, 20.0889...","[19.48900032043457, 19.48900032043457, 19.4890...","[18.538999557495117, 18.538999557495117, 18.53...","[0.04800000041723251, 0.04699999839067459, 0.0...","[0.16099999845027924, 0.16099999845027924, 0.1...","[0.22300000488758087, 0.22300000488758087, 0.2...","[0.2590000033378601, 0.2590000033378601, 0.259...",[ 7.37670809e-02 7.41081908e-02 7.28954226e-...
1,Flash Flood,2014-09-21 12:30:00,MST-7,2014-09-21 23:30:00,33.56,-105.58,33.44,-104.62,1,2014-09-21,...,"[20.51999855041504, 20.880001068115234, 19.440...","[19.10249900817871, 18.502500534057617, 18.352...","[16.65250015258789, 16.802499771118164, 16.952...","[17.502500534057617, 17.502500534057617, 17.50...","[17.702499389648438, 17.702499389648438, 17.70...","[0.4050000011920929, 0.4050000011920929, 0.405...","[0.3840000033378601, 0.3840000033378601, 0.384...","[0.1770000010728836, 0.1770000010728836, 0.177...","[0.1589999943971634, 0.1589999943971634, 0.158...",[ 0.10417991 0.10618292 0.10155834 0.102084...
2,Flash Flood,2014-09-27 09:30:00,MST-7,2014-09-27 20:00:00,37.54,-112.86,37.66,-114.04,1,2014-09-27,...,"[32.39999771118164, 37.07999801635742, 37.7999...","[18.75950050354004, 21.00950050354004, 22.8094...","[14.459500312805176, 14.759500503540039, 15.15...","[15.559499740600586, 15.559499740600586, 15.55...","[14.009500503540039, 14.009500503540039, 14.00...","[0.15399999916553497, 0.15199999511241913, 0.1...","[0.15800000727176666, 0.15800000727176666, 0.1...","[0.18700000643730164, 0.18700000643730164, 0.1...","[0.26100000739097595, 0.26100000739097595, 0.2...",[ 6.16712943e-02 6.11631535e-02 6.78475946e-...
3,Flash Flood,2014-09-09 17:40:00,CST-6,2014-09-10 05:15:00,41.50,-93.68,41.50,-93.66,1,2014-09-09,...,"[10.799999237060547, 5.039999961853027, 4.6799...","[21.804500579833984, 20.10449981689453, 18.554...","[20.954500198364258, 20.90450096130371, 20.754...","[21.35449981689453, 21.35449981689453, 21.3045...","[17.954500198364258, 17.954500198364258, 17.95...","[0.3059999942779541, 0.3059999942779541, 0.305...","[0.2930000126361847, 0.2930000126361847, 0.293...","[0.2460000067949295, 0.2460000067949295, 0.246...","[0.31700000166893005, 0.31700000166893005, 0.3...",[ 8.60121250e-02 8.10814425e-02 8.40162709e-...
4,Flash Flood,2014-09-09 18:47:00,CST-6,2014-09-10 05:00:00,40.94,-94.38,40.90,-94.38,1,2014-09-10,...,"[7.559999465942383, 9.359999656677246, 10.0799...","[19.87150001525879, 18.42150115966797, 17.3715...","[20.821500778198242, 20.67150115966797, 20.471...","[21.821500778198242, 21.821500778198242, 21.82...","[18.821500778198242, 18.821500778198242, 18.82...","[0.328000009059906, 0.3269999921321869, 0.3269...","[0.30399999022483826, 0.30399999022483826, 0.3...","[0.2240000069141388, 0.2240000069141388, 0.224...","[0.30300000309944153, 0.30300000309944153, 0.3...",[ 1.00482784e-01 9.88258719e-02 9.94119495e-...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
869,Flood,2020-05-05 07:30:00,EST-5,2020-05-05 12:30:00,37.14,-84.68,37.14,-84.68,0,2020-05-05,...,"[18.719999313354492, 21.239999771118164, 28.44...","[10.500499725341797, 11.70050048828125, 13.050...","[11.05049991607666, 11.05049991607666, 11.1505...","[11.000499725341797, 11.000499725341797, 11.00...","[10.000499725341797, 10.000499725341797, 10.00...","[0.462999999

In [ ]:
import numpy as np

# Assuming final_df_images_features["image_features"].iloc[0] contains the string
final_df
def apply_string_cleaning(array_str):
    # Clean the string by removing unwanted characters
    try:
        print(type(array_str))
        cleaned_str = array_str.replace('[', '').replace(']', '').replace('\n', ' ')
        array = np.fromstring(cleaned_str, sep=' ')
        return array
    except Exception as e:
        print(array_str)
# Clean the string by removing unwanted characters
final_df["images_features"] = final_df["images_features"].apply(apply_string_cleaning)

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class

C:\Users\naman\AppData\Local\Temp\ipykernel_35100\3177189297.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df["images_features"] = final_df["images_features"].apply(apply_string_cleaning)


In [ ]:
tqdm.pandas()
# Expand the image features into separate columns (each feature as a column)
def expand_features(feat_array):
    if isinstance(feat_array, np.ndarray):
        return pd.Series(feat_array)
    else:
        return pd.Series()

df_image_features = df['image_features'].progress_apply(expand_features)
df_image_features.columns = [f'img_feat_{i}' for i in range(df_image_features.shape[1])]

# ==========================
# 5. COMBINE IMAGE FEATURES WITH OTHER DATA
# ==========================
# Drop columns not needed for modeling
cols_to_drop = ['event_type', 'begin_date_time', 'cz_timezone', 'end_date_time',
                'begin_date_utc', 'begin_time_utc', 'end_date_utc', 'end_time_utc',
                'start_date_72h', 'end_date', 'event_datetime', 'prev_72h_weather',
                'filenames', 'image_features']
df_model = df.drop(columns=cols_to_drop, errors='ignore')

# Concatenate the expanded image features with the rest of the data
df_model = pd.concat([df_model, df_image_features], axis=1)

100%|██████████| 874/874 [00:00<00:00, 6231.20it/s]


In [ ]:
def safe_literal_eval(x):
    if isinstance(x, list):
        x = [float(i) for i in x]
        return x
    try:
        temp = ast.literal_eval(x)
        temp = [float(i) for i in temp]
        return temp

    except Exception:
        return np.nan

# Identify all columns starting with "prev_"
prev_cols = [col for col in df_model.columns if col.startswith('prev_')]
for col in prev_cols:
    df_model[col] = df_model[col].apply(safe_literal_eval)
print(prev_cols)
print(df_model["prev_72h_temperature_2m"].values)
# Aggregate each "prev_" column into summary statistics: mean, std, min, and max.
for col in prev_cols:
    df_model[f'{col}_mean'] = df_model[col].apply(lambda x: np.mean(x) if isinstance(x, list) else np.nan)
    df_model[f'{col}_std'] = df_model[col].apply(lambda x: np.std(x) if isinstance(x, list) else np.nan)
    df_model[f'{col}_min'] = df_model[col].apply(lambda x: np.min(x) if isinstance(x, list) else np.nan)
    df_model[f'{col}_max'] = df_model[col].apply(lambda x: np.max(x) if isinstance(x, list) else np.nan)

# Drop the raw list columns to simplify the data
df_model.drop(columns=prev_cols, inplace=True)

# Drop columns that are non-numeric or not needed for modeling
cols_to_drop = ['event_type', 'begin_date_time', 'cz_timezone', 'end_date_time',
                'begin_date_utc', 'begin_time_utc', 'end_date_utc', 'end_time_utc',
                'start_date_72h', 'end_date', 'event_datetime', 'prev_72h_weather',
                'filenames']
df_model = df_model.drop(columns=cols_to_drop, errors='ignore')

# Separate features (X) and target (y). Here, 'extreme' is the target.
X = df_model.drop(columns=['extreme', 'cluster', 'event_id'], errors='ignore')
y = df_model['extreme']

# Fill any missing values with the column mean.
X = X.fillna(X.mean())

['prev_72h_temperature_2m', 'prev_72h_relative_humidity_2m', 'prev_72h_dew_point_2m', 'prev_72h_apparent_temperature', 'prev_72h_precipitation', 'prev_72h_rain', 'prev_72h_snowfall', 'prev_72h_snow_depth', 'prev_72h_weather_code', 'prev_72h_pressure_msl', 'prev_72h_surface_pressure', 'prev_72h_cloud_cover', 'prev_72h_cloud_cover_low', 'prev_72h_cloud_cover_mid', 'prev_72h_cloud_cover_high', 'prev_72h_et0_fao_evapotranspiration', 'prev_72h_vapour_pressure_deficit', 'prev_72h_wind_speed_10m', 'prev_72h_wind_speed_100m', 'prev_72h_wind_direction_10m', 'prev_72h_wind_direction_100m', 'prev_72h_wind_gusts_10m', 'prev_72h_soil_temperature_0_to_7cm', 'prev_72h_soil_temperature_7_to_28cm', 'prev_72h_soil_temperature_28_to_100cm', 'prev_72h_soil_temperature_100_to_255cm', 'prev_72h_soil_moisture_0_to_7cm', 'prev_72h_soil_moisture_7_to_28cm', 'prev_72h_soil_moisture_28_to_100cm', 'prev_72h_soil_moisture_100_to_255cm']
[list([25.18899917602539, 24.388999938964844, 22.23900032043457, 20.8389987945

C:\Users\naman\AppData\Local\Temp\ipykernel_39452\1099552111.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_model[f'{col}_max'] = df_model[col].apply(lambda x: np.max(x) if isinstance(x, list) else np.nan)
C:\Users\naman\AppData\Local\Temp\ipykernel_39452\1099552111.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_model[f'{col}_mean'] = df_model[col].apply(lambda x: np.mean(x) if isinstance(x, list) else np.nan)
C:\Users\naman\AppData\Local\Temp\ipykernel_39452\1099552111.py:22: PerformanceWarning: DataFrame is h

In [ ]:


# ==========================
# 6. TRAINING WITH A SKLEARN/IMBLEARN PIPELINE
# ==========================
from sklearn.impute import SimpleImputer
from imblearn.pipeline import Pipeline as imbpipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.metrics import (roc_auc_score, balanced_accuracy_score, f1_score, 
                             precision_score, recall_score, confusion_matrix, make_scorer)

logging.info("Starting train-test split.")
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, 
                                                    test_size=0.2, random_state=42)
logging.info("Completed train-test split.")

# Build a pipeline for modeling
pipeline = imbpipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('sampler', SMOTE(random_state=42)),
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

scoring = {
    'roc_auc': 'roc_auc',
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
    'f1': 'f1',
    'precision': 'precision',
    'recall': 'recall'
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
logging.info("Starting cross-validation.")
cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True)
logging.info("Completed cross-validation.")

for key in sorted(cv_results.keys()):
    logging.info(f"{key}: {np.mean(cv_results[key]):.4f}")

param_grid = {
    'clf__n_estimators': [100, 200],
    'clf__max_depth': [None, 10, 20]
}

logging.info("Starting hyperparameter tuning with GridSearchCV.")
grid = GridSearchCV(pipeline, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)
logging.info("Completed hyperparameter tuning.")

logging.info(f"Best hyperparameters: {grid.best_params_}")
logging.info(f"Best CV ROC AUC: {grid.best_score_:.4f}")

best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_prob)
bal_acc = balanced_accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

logging.info(f"Test Set ROC AUC: {roc_auc:.4f}")
logging.info(f"Test Set Balanced Accuracy: {bal_acc:.4f}")
logging.info(f"Test Set F1 Score: {f1:.4f}")
logging.info(f"Test Set Precision: {precision:.4f}")
logging.info(f"Test Set Recall: {recall:.4f}")
logging.info(f"Confusion Matrix:\n{cm}")


2025-03-13 11:56:59,755 - INFO - Starting train-test split.
2025-03-13 11:56:59,787 - INFO - Completed train-test split.
2025-03-13 11:56:59,789 - INFO - Starting cross-validation.
c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 501, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\naman\AppData\Local\Programs\Python\Python3

In [5]:
df.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_soil_moisture_7_to_28cm_min,prev_72h_soil_moisture_7_to_28cm_max,prev_72h_soil_moisture_28_to_100cm_mean,prev_72h_soil_moisture_28_to_100cm_std,prev_72h_soil_moisture_28_to_100cm_min,prev_72h_soil_moisture_28_to_100cm_max,prev_72h_soil_moisture_100_to_255cm_mean,prev_72h_soil_moisture_100_to_255cm_std,prev_72h_soil_moisture_100_to_255cm_min,prev_72h_soil_moisture_100_to_255cm_max
4,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,...,0.161,0.161,0.2230,0.000000,0.223,0.223,0.259,0.0,0.259,0.259
10,Flash Flood,2014-09-21 12:30:00,MST-7,2014-09-21 23:30:00,33.56,-105.58,33.44,-104.62,1,2014-09-21,...,0.382,0.384,0.1780,0.000775,0.177,0.179,0.159,0.0,0.159,0.159
12,Flash Flood,2014-09-27 09:30:00,MST-7,2014-09-27 20:00:00,37.54,-112.86,37.66,-114.04,1,2014-09-27,...,0.158,0.158,0.1863,0.000458,0.186,0.187,0.261,0.0,0.261,0.261
18,Flash Flood,2014-09-09 17:40:00,CST-6,2014-09-10 05:15:00,41.50,-93.68,41.50,-93.66,1,2014-09-09,...,0.293,0.293,0.2460,0.000000,0.246,0.246,0.317,0.0,0.317,0.317
21,Flash Flood,2014-09-09 18:47:00,CST-6,2014-09-10 05:00:00,40.94,-94.38,40.90,-94.38,1,2014-09-10,...,0.304,0.304,0.2240,0.000000,0.224,0.224,0.303,0.0,0.303,0.303


In [1]:
import pandas as pd
final_df_images_features = pd.read_csv("final_df_image_features.csv")

In [2]:
final_df_images_features.head()

,event_type,begin_date_time,cz_timezone,end_date_time,begin_lat,begin_lon,end_lat,end_lon,extreme,begin_date_utc,...,prev_72h_wind_gusts_10m,prev_72h_soil_temperature_0_to_7cm,prev_72h_soil_temperature_7_to_28cm,prev_72h_soil_temperature_28_to_100cm,prev_72h_soil_temperature_100_to_255cm,prev_72h_soil_moisture_0_to_7cm,prev_72h_soil_moisture_7_to_28cm,prev_72h_soil_moisture_28_to_100cm,prev_72h_soil_moisture_100_to_255cm,image_features
0,Flash Flood,2014-09-27 16:00:00,MST-7,2014-09-27 23:00:00,39.70,-110.88,39.54,-110.92,1,2014-09-27,...,"['26.639999389648438', '24.119998931884766', '...","['30.48900032043457', '28.388999938964844', '2...","['19.68899917602539', '19.93899917602539', '20...","['19.48900032043457', '19.48900032043457', '19...","['18.538999557495117', '18.538999557495117', '...","['0.04800000041723251', '0.04699999839067459',...","['0.16099999845027924', '0.16099999845027924',...","['0.22300000488758087', '0.22300000488758087',...","['0.2590000033378601', '0.2590000033378601', '...",[ 7.37670809e-02 7.41081908e-02 7.28954226e-...
1,Flash Flood,2014-09-21 12:30:00,MST-7,2014-09-21 23:30:00,33.56,-105.58,33.44,-104.62,1,2014-09-21,...,"['20.51999855041504', '20.880001068115234', '1...","['19.10249900817871', '18.502500534057617', '1...","['16.65250015258789', '16.802499771118164', '1...","['17.502500534057617', '17.502500534057617', '...","['17.702499389648438', '17.702499389648438', '...","['0.4050000011920929', '0.4050000011920929', '...","['0.3840000033378601', '0.3840000033378601', '...","['0.1770000010728836', '0.1770000010728836', '...","['0.1589999943971634', '0.1589999943971634', '...",[ 0.10417991 0.10618292 0.10155834 0.102084...
2,Flash Flood,2014-09-27 09:30:00,MST-7,2014-09-27 20:00:00,37.54,-112.86,37.66,-114.04,1,2014-09-27,...,"['32.39999771118164', '37.07999801635742', '37...","['18.75950050354004', '21.00950050354004', '22...","['14.459500312805176', '14.759500503540039', '...","['15.559499740600586', '15.559499740600586', '...","['14.009500503540039', '14.009500503540039', '...","['0.15399999916553497', '0.15199999511241913',...","['0.15800000727176666', '0.15800000727176666',...","['0.18700000643730164', '0.18700000643730164',...","['0.26100000739097595', '0.26100000739097595',...",[ 6.16712943e-02 6.11631535e-02 6.78475946e-...
3,Flash Flood,2014-09-09 17:40:00,CST-6,2014-09-10 05:15:00,41.50,-93.68,41.50,-93.66,1,2014-09-09,...,"['10.799999237060547', '5.039999961853027', '4...","['21.804500579833984', '20.10449981689453', '1...","['20.954500198364258', '20.90450096130371', '2...","['21.35449981689453', '21.35449981689453', '21...","['17.954500198364258', '17.954500198364258', '...","['0.3059999942779541', '0.3059999942779541', '...","['0.2930000126361847', '0.2930000126361847', '...","['0.2460000067949295', '0.2460000067949295', '...","['0.31700000166893005', '0.31700000166893005',...",[ 8.60121250e-02 8.10814425e-02 8.40162709e-...
4,Flash Flood,2014-09-09 18:47:00,CST-6,2014-09-10 05:00:00,40.94,-94.38,40.90,-94.38,1,2014-09-10,...,"['7.559999465942383', '9.359999656677246', '10...","['19.87150001525879', '18.42150115966797', '17...","['20.821500778198242', '20.67150115966797', '2...","['21.821500778198242', '21.821500778198242', '...","['18.821500778198242', '18.821500778198242', '...","['0.328000009059906', '0.3269999921321869', '0...","['0.30399999022483826', '0.30399999022483826',...","['0.2240000069141388', '0.2240000069141388', '...","['0.30300000309944153', '0.30300000309944153',...",[ 1.00482784e-01 9.88258719e-02 9.94119495e-...


In [92]:
final_df_images_features["prev_72h_apparent_temperature"].iloc[0]

"['21.92059326171875', '21.165401458740234', '19.553081512451172', '18.615144729614258', '18.25693130493164', '17.188142776489258', '16.315061569213867', '15.545417785644531', '14.340232849121094', '13.456636428833008']"

In [ ]:
df["prev_72h_apparent_temperature"]

0      ['21.92059326171875', '21.165401458740234', '1...
1      ['18.73444175720215', '17.883811950683594', '1...
2      ['16.434614181518555', '18.342021942138672', '...
3      ['21.209449768066406', '19.79183006286621', '1...
4      ['17.835126876831055', '15.634904861450195', '...
                             ...                        
869    ['11.189874649047852', '14.947141647338867', '...
870    ['16.275737762451172', '18.222501754760742', '...
871    ['8.876077651977539', '9.038336753845215', '9....
872    ['3.6160593032836914', '4.919330596923828', '5...
873    ['26.172447204589844', '25.998275756835938', '...
Name: prev_72h_apparent_temperature, Length: 874, dtype: object

## temporal fusion

In [42]:
temp.drop(columns=["prev_72h_weather"], inplace=True)

In [25]:
temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 874 entries, 0 to 873
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   event_type                              874 non-null    object 
 1   begin_date_time                         874 non-null    object 
 2   cz_timezone                             874 non-null    object 
 3   end_date_time                           874 non-null    object 
 4   begin_lat                               874 non-null    float64
 5   begin_lon                               874 non-null    float64
 6   end_lat                                 874 non-null    float64
 7   end_lon                                 874 non-null    float64
 8   extreme                                 874 non-null    int64  
 9   begin_date_utc                          874 non-null    object 
 10  begin_time_utc                          874 non-null    object

: 

In [ ]:


# ==========================
# 1. LOAD AND PREPROCESS DATA
# ==========================
# Load your 

# Apply restructuring
restructured_df = restructure_dataframe(df)
# ==========================
# 2. PREPARE TIME-SERIES DATASET FOR TFT
# ==========================
# Define time-series dataset for Temporal Fusion Transformer
# Correcting the TimeSeriesDataSet initialization



# Split dataset into training and validation sets
train_data, val_data = data.split_by_time()

train_dataloader = train_data.to_dataloader(train=True, batch_size=64)
val_dataloader = val_data.to_dataloader(train=False, batch_size=64)

# ==========================
# 3. TRAIN TEMPORAL FUSION TRANSFORMER MODEL
# ==========================
# Initialize Temporal Fusion Transformer model
tft = TemporalFusionTransformer.from_dataset(
    train_data,
    loss=QuantileLoss(),         # Quantile loss for probabilistic forecasting
    hidden_size=128,             # Hidden size of the model
    attention_head_size=4,       # Number of attention heads
    dropout=0.1,                 # Dropout rate to prevent overfitting
)

# Define optimizer and trainer settings
optimizer = torch.optim.Adam(tft.parameters(), lr=1e-3)

for epoch in range(30):  # Train for 30 epochs
    tft.train()
    epoch_loss = []
    for batch in train_dataloader:
        optimizer.zero_grad()
        loss = tft.training_step(batch)
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    
    print(f"Epoch {epoch + 1}: Loss = {np.mean(epoch_loss):.4f}")

# ==========================
# 4. EVALUATION METRICS AND PREDICTIONS
# ==========================
tft.eval()
val_predictions = []
val_targets = []
with torch.no_grad():
    for batch in val_dataloader:
        predictions = tft.predict(batch)
        targets = batch["target"].numpy()
        val_predictions.append(predictions.numpy())
        val_targets.append(targets)

val_predictions = np.concatenate(val_predictions)
val_targets = np.concatenate(val_targets)

roc_auc = roc_auc_score(val_targets, val_predictions)
balanced_acc = balanced_accuracy_score(val_targets, val_predictions.round())

print(f"Validation ROC AUC: {roc_auc:.4f}")
print(f"Validation Balanced Accuracy: {balanced_acc:.4f}")


c:\Users\naman\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_forecasting\models\base_model.py:27: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


ValueError: could not convert string to float: '['

In [155]:
temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 874 entries, 0 to 873
Data columns (total 50 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   event_type                              874 non-null    object 
 1   begin_date_time                         874 non-null    object 
 2   cz_timezone                             874 non-null    object 
 3   end_date_time                           874 non-null    object 
 4   begin_lat                               874 non-null    float64
 5   begin_lon                               874 non-null    float64
 6   end_lat                                 874 non-null    float64
 7   end_lon                                 874 non-null    float64
 8   extreme                                 874 non-null    int64  
 9   begin_date_utc                          874 non-null    object 
 10  begin_time_utc                          874 non-null    object

In [156]:

# Restructure dataframe
def restructure_dataframe(df):
    rows = []
    
    # Identify all columns with the `prev_` prefix
    prev_columns = [col for col in df.columns if col.startswith('prev_')]
    
    for _, row in df.iterrows():
        # Extract static features
        static_features = {
            'event_id': row['event_id'],
            'event_type': row['event_type'],
            'begin_date_time': row['begin_date_time'],
            'image_features': np.array(row['image_features']),
            'extreme': row['extreme']
        }
        
        # Expand all `prev_` columns into individual rows
        num_timesteps = 10  # Assume all `prev_` columns have the same length (72 hours)
        
        for time_idx in range(num_timesteps):
            expanded_row = {**static_features, 'time_idx': time_idx}
            
            # Add values from each `prev_` column for the current timestep
            for col in prev_columns:
                expanded_row[col.replace('prev_', '')] = float(row[col][time_idx])
            
            rows.append(expanded_row)
    
    return pd.DataFrame(rows)

# Apply restructuring
restructured_df = restructure_dataframe(temp)

# Display restructured dataframe
print(restructured_df.head())

   event_id   event_type      begin_date_time  \
0         4  Flash Flood  2014-09-27 16:00:00   
1         4  Flash Flood  2014-09-27 16:00:00   
2         4  Flash Flood  2014-09-27 16:00:00   
3         4  Flash Flood  2014-09-27 16:00:00   
4         4  Flash Flood  2014-09-27 16:00:00   

                                      image_features  extreme  time_idx  \
0  [ 7.37670809e-02  7.41081908e-02  7.28954226e-...        1         0   
1  [ 7.37670809e-02  7.41081908e-02  7.28954226e-...        1         1   
2  [ 7.37670809e-02  7.41081908e-02  7.28954226e-...        1         2   
3  [ 7.37670809e-02  7.41081908e-02  7.28954226e-...        1         3   
4  [ 7.37670809e-02  7.41081908e-02  7.28954226e-...        1         4   

   72h_temperature_2m  72h_relative_humidity_2m  72h_dew_point_2m  \
0           25.188999                 22.601961             2.339   
1           24.389000                 23.206690             2.039   
2           22.239000                 27.090515

In [157]:
restructured_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8740 entries, 0 to 8739
Data columns (total 36 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   event_id                           8740 non-null   int64  
 1   event_type                         8740 non-null   object 
 2   begin_date_time                    8740 non-null   object 
 3   image_features                     8740 non-null   object 
 4   extreme                            8740 non-null   int64  
 5   time_idx                           8740 non-null   int64  
 6   72h_temperature_2m                 8740 non-null   float64
 7   72h_relative_humidity_2m           8740 non-null   float64
 8   72h_dew_point_2m                   8740 non-null   float64
 9   72h_apparent_temperature           8740 non-null   float64
 10  72h_precipitation                  8740 non-null   float64
 11  72h_rain                           8740 non-null   float

In [65]:
import ast

def convert_to_array(val):
    if isinstance(val, str):
        return np.array(ast.literal_eval(val))
    return val

df['image_features'] = df['image_features'].apply(convert_to_array)


In [100]:
restructured_df.columns

Index(['event_type', 'begin_date_time', 'image_features', 'extreme',
       'time_idx', '72h_temperature_2m', '72h_relative_humidity_2m',
       '72h_dew_point_2m', '72h_apparent_temperature', '72h_precipitation',
       '72h_rain', '72h_snowfall', '72h_snow_depth', '72h_weather_code',
       '72h_pressure_msl', '72h_surface_pressure', '72h_cloud_cover',
       '72h_cloud_cover_low', '72h_cloud_cover_mid', '72h_cloud_cover_high',
       '72h_et0_fao_evapotranspiration', '72h_vapour_pressure_deficit',
       '72h_wind_speed_10m', '72h_wind_speed_100m', '72h_wind_direction_10m',
       '72h_wind_direction_100m', '72h_wind_gusts_10m',
       '72h_soil_temperature_0_to_7cm', '72h_soil_temperature_7_to_28cm',
       '72h_soil_temperature_28_to_100cm', '72h_soil_temperature_100_to_255cm',
       '72h_soil_moisture_0_to_7cm', '72h_soil_moisture_7_to_28cm',
       '72h_soil_moisture_28_to_100cm', '72h_soil_moisture_100_to_255cm'],
      dtype='object')

In [158]:
def format_images(t):
    t = t.tolist()
    t = re.sub(r'[\n\s]+', ' ', t)
    t = t.replace("[", "").replace("]", "").replace("\n", "")
    return np.fromstring(t, sep=' ')
restructured_df["image_features"] = restructured_df["image_features"].apply(format_images)

In [85]:
df_model["combined_features"].iloc[0]

array(['25.18899917602539', '22.601961135864258', '2.3389999866485596',
       '21.92059326171875', '0.0', '0.0', '0.0', '0.0', '0.0',
       '1015.0999755859375', '818.5861206054688', '0.0', '0.0', '0.0',
       '0.0', '0.39395228028297424', '2.4806699752807617',
       '10.495713233947754', '12.682018280029297', '174.0939483642578',
       '173.4803009033203', '26.639999389648438', '30.48900032043457',
       '19.68899917602539', '19.48900032043457', '18.538999557495117',
       '0.04800000041723251', '0.16099999845027924',
       '0.22300000488758087', '0.2590000033378601',
       '[ 7.37670809e-02  7.41081908e-02  7.28954226e-02  6.94682375e-02\n  6.56651929e-02  6.68598935e-02  6.34842962e-02  6.25604838e-02\n  6.22117855e-02  6.08440749e-02  5.99260330e-02  5.92180975e-02\n  5.97874597e-02  6.51937351e-02  6.59779310e-02  6.17594793e-02\n  6.10737503e-02  5.90772219e-02  5.68338819e-02  5.47151566e-02\n  5.27159646e-02  5.35173416e-02  5.22577353e-02  5.83954118e-02\n  5.89565635

In [159]:
import pandas as pd
import numpy as np
import torch
from pytorch_forecasting import TemporalFusionTransformer
from pytorch_forecasting.data import TimeSeriesDataSet
from pytorch_forecasting.metrics import QuantileLoss
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

df_model = restructured_df.copy()
# Combine temporal hourly features and image features into one array per row
hourly_features = [col for col in df_model.columns if "72h_" in col]
def combine_features(row):
    
    hourly_data = np.array([row[col] for col in hourly_features])
   
    image_data = row["image_features"]

    
    return np.concatenate([hourly_data, image_data])
df_model["combined_features"] = df_model.apply(combine_features, axis=1)


In [153]:
df_model.to_csv("df_model.csv", index=False)

In [160]:
df_model.head()

,event_id,event_type,begin_date_time,image_features,extreme,time_idx,72h_temperature_2m,72h_relative_humidity_2m,72h_dew_point_2m,72h_apparent_temperature,...,72h_wind_gusts_10m,72h_soil_temperature_0_to_7cm,72h_soil_temperature_7_to_28cm,72h_soil_temperature_28_to_100cm,72h_soil_temperature_100_to_255cm,72h_soil_moisture_0_to_7cm,72h_soil_moisture_7_to_28cm,72h_soil_moisture_28_to_100cm,72h_soil_moisture_100_to_255cm,combined_features
0,4,Flash Flood,2014-09-27 16:00:00,"[0.0737670809, 0.0741081908, 0.0728954226, 0.0...",1,0,25.188999,22.601961,2.339,21.920593,...,26.639999,30.489000,19.688999,19.489,18.539,0.048,0.161,0.223,0.259,"[25.18899917602539, 22.601961135864258, 2.3389..."
1,4,Flash Flood,2014-09-27 16:00:00,"[0.0737670809, 0.0741081908, 0.0728954226, 0.0...",1,1,24.389000,23.206690,2.039,21.165401,...,24.119999,28.389000,19.938999,19.489,18.539,0.047,0.161,0.223,0.259,"[24.388999938964844, 23.206689834594727, 2.039..."
2,4,Flash Flood,2014-09-27 16:00:00,"[0.0737670809, 0.0741081908, 0.0728954226, 0.0...",1,2,22.239000,27.090515,2.389,19.553082,...,19.799999,25.088999,20.088999,19.489,18.539,0.046,0.161,0.223,0.259,"[22.23900032043457, 27.09051513671875, 2.38899..."
3,4,Flash Flood,2014-09-27 16:00:00,"[0.0737670809, 0.0741081908, 0.0728954226, 0.0...",1,3,20.838999,29.200331,2.239,18.615145,...,8.640000,21.588999,20.139000,19.489,18.539,0.046,0.161,0.223,0.259,"[20.838998794555664, 29.20033073425293, 2.2390..."
4,4,Flash Flood,2014-09-27 16:00:00,"[0.0737670809, 0.0741081908, 0.0728954226, 0.0...",1,4,19.938999,31.758774,2.639,18.256931,...,4.320000,18.938999,20.088999,19.489,18.539,0.046,0.161,0.223,0.259,"[19.93899917602539, 31.758773803710938, 2.6389..."


In [165]:
df_model["event_id"]=df_model["event_id"].apply(lambda x: str(x))

In [169]:
import pandas as pd
import torch
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.data import GroupNormalizer
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping

# Load your DataFrame (assuming it's already preprocessed)
df = df_model.copy() # Replace with your actual DataFrame file path

from pytorch_forecasting.data import TimeSeriesDataSet
# Fill missing timesteps for each group
df = df.groupby("event_type").apply(
    lambda group: group.set_index("time_idx")
    .reindex(range(group["time_idx"].min(), group["time_idx"].max() + 1))
    .reset_index()
).reset_index(drop=True)

# Fill missing values with appropriate defaults (e.g., 0 or NaN)
df.fillna(0, inplace=True)  # Replace with appropriate default values for your dataset

# Define parameters for the TimeSeriesDataSet
max_encoder_length = 10  # Lookback period
max_prediction_length = 1  # Prediction horizon (extreme event classification)
batch_size = 64  # Batch size for training

# Create a TimeSeriesDataSet object
dataset = TimeSeriesDataSet(
    df,
    time_idx="time_idx",  # Column indicating the time step index
    target="extreme",  # Target column (binary classification: extreme or not)
    group_ids=["event_type"],  # Grouping by event type
    max_encoder_length=max_encoder_length,
    max_prediction_length=max_prediction_length,
    static_categoricals=["event_type"],  # Static categorical features
    time_varying_known_reals=[
        "72h_temperature_2m",
        "72h_relative_humidity_2m",
        "72h_dew_point_2m",
        "72h_apparent_temperature",
        "72h_precipitation",
        "72h_soil_temperature_28_to_100cm",
        "72h_soil_temperature_100_to_255cm",
        "72h_soil_moisture_0_to_7cm",
        "72h_soil_moisture_7_to_28cm",
        "72h_soil_moisture_28_to_100cm",
        "72h_soil_moisture_100_to_255cm",
    ],  # Features that are known at prediction time
    time_varying_unknown_reals=[],  # Add unknown features if applicable
    add_relative_time_idx=True,  # Add relative time index as a feature
    add_target_scales=True,  # Scale target variable
    add_encoder_length=True,  # Add encoder length as a feature
)




ValueError: cannot reindex on an axis with duplicate labels

In [168]:
# Inspect the time_idx column
time_idx_summary = {
    'min_time_idx': df['time_idx'].min(),
    'max_time_idx': df['time_idx'].max(),
    'unique_time_idx_count': df['time_idx'].nunique(),
    'time_idx_gaps': df['time_idx'].diff().value_counts().to_dict()
}
print(time_idx_summary)


{'min_time_idx': np.int64(0), 'max_time_idx': np.int64(9), 'unique_time_idx_count': 10, 'time_idx_gaps': {1.0: 7866, -9.0: 873}}


In [170]:
# Check for duplicates in time_idx within each event_type group
duplicates = df.groupby("event_type")["time_idx"].apply(lambda x: x.duplicated()).sum()
print(f"Number of duplicate time_idx values: {duplicates}")


Number of duplicate time_idx values: 8690
